# DR9 Fixed-Radius Binned Spearman Correlations

This notebook visualizes the fixed-radius local environment estimator using the preferred signal annulus
\(1.5 < R < 7\,h^{-1}\mathrm{Mpc}\). It bins clusters by richness and redshift, then measures the Spearman rank correlation between richness and the local overdensity statistic in each bin.

The notebook is visualization-only: it reads the output of the radius-grid MPI run, `radius_grid_overdensity/rm_dr9_radius_grid_overdensity.fits`, and filters to the preferred radius definition.

In [ ]:
# Package imports and plotting defaults
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from astropy.table import Table
from astropy.units import UnitsWarning
from scipy.stats import spearmanr, pearsonr

warnings.filterwarnings(
    "ignore",
    message=r".*did not parse as fits unit.*",
    category=UnitsWarning,
)

sns.set_context("talk")
sns.set_style("ticks")
mpl.rcParams["figure.dpi"] = 120
mpl.rcParams["savefig.dpi"] = 180
mpl.rcParams["axes.linewidth"] = 1.1
mpl.rcParams["xtick.direction"] = "in"
mpl.rcParams["ytick.direction"] = "in"
mpl.rcParams["xtick.top"] = True
mpl.rcParams["ytick.right"] = True

In [ ]:
# Paths and analysis configuration
# The relative-path fallback is useful when running from the repository root.
try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name == "local_overdensity":
    REPO_ROOT = NOTEBOOK_DIR.parent
else:
    REPO_ROOT = Path("/global/homes/z/zzhang13/DESI/Projection")
    if not REPO_ROOT.exists():
        REPO_ROOT = Path.cwd()

OUTPUT_DIR = REPO_ROOT / "local_overdensity" / "dr9_outputs" / "radius_grid_overdensity"
TABLE_PATH = OUTPUT_DIR / "rm_dr9_radius_grid_overdensity.fits"
PLOT_DIR = OUTPUT_DIR / "binned_spearman_plots_1p5_7"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

Z_COL = "Z_SPEC_x"
Y_COL = "Sigma_excess"   # alternatives: "N_excess", "Sigma_signal_covcorr"

R_SIG_IN_HMPC = 1.5
R_SIG_OUT_HMPC = 7.0
R_BG_IN_HMPC = 7.0
R_BG_OUT_HMPC = 12.0

Z_BINS = [(0.1, 0.2), (0.2, 0.3), (0.3, 0.4)]

# Richness-bin choices. Adjust these after inspecting the bin counts below.
LAMBDA_BINS_RM = np.array([20, 25, 30, 40, 100], dtype=float)
LAMBDA_BINS_SPEC = np.array([5, 10, 20, 40, 70, 120, 200, 300], dtype=float)

# Optional coverage requirement for the correlation measurement.
# Set to 0.8 if you want the same kind of conservative coverage cut as earlier diagnostics.
COVERAGE_MIN = 0.0

N_BOOT = 1000
RNG_SEED = 123

print("Repo root:", REPO_ROOT)
print("Input table:", TABLE_PATH)
print("Plot dir:", PLOT_DIR)

In [ ]:
# Load fixed-radius local-overdensity table
if not TABLE_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {TABLE_PATH}. Run local_overdensity/DR9_radius_grid_overdensity_mpi.py first."
    )

radius_grid_table = Table.read(TABLE_PATH)

def _col_float_load(tab, col):
    arr = np.ma.asarray(tab[col], dtype=float)
    return np.ma.filled(arr, np.nan)

required_radius_cols = ["r_sig_in_hmpc", "r_sig_out_hmpc", "r_bg_in_hmpc", "r_bg_out_hmpc"]
missing_radius_cols = [col for col in required_radius_cols if col not in radius_grid_table.colnames]
if missing_radius_cols:
    raise KeyError(f"Radius-grid table is missing radius-definition columns: {missing_radius_cols}")

radius_mask = (
    np.isclose(_col_float_load(radius_grid_table, "r_sig_in_hmpc"), R_SIG_IN_HMPC)
    & np.isclose(_col_float_load(radius_grid_table, "r_sig_out_hmpc"), R_SIG_OUT_HMPC)
    & np.isclose(_col_float_load(radius_grid_table, "r_bg_in_hmpc"), R_BG_IN_HMPC)
    & np.isclose(_col_float_load(radius_grid_table, "r_bg_out_hmpc"), R_BG_OUT_HMPC)
)

if np.count_nonzero(radius_mask) == 0:
    available = radius_grid_table[required_radius_cols].to_pandas().drop_duplicates().sort_values(required_radius_cols)
    display(available)
    raise ValueError(
        "No rows matched the requested radius definition: "
        f"signal {R_SIG_IN_HMPC}-{R_SIG_OUT_HMPC}, background {R_BG_IN_HMPC}-{R_BG_OUT_HMPC} h^-1 Mpc"
    )

table = radius_grid_table[radius_mask]
print(f"Loaded radius-grid rows: {len(radius_grid_table):,}")
print(f"Rows after radius filter: {len(table):,}")
print(
    f"Using signal annulus {R_SIG_IN_HMPC:g} < R < {R_SIG_OUT_HMPC:g} h^-1 Mpc "
    f"and background annulus {R_BG_IN_HMPC:g} < R < {R_BG_OUT_HMPC:g} h^-1 Mpc"
)
print(table)
print("Columns:")
print(table.colnames)

In [ ]:
# Helper functions
def col_float(tab, col):
    arr = np.ma.asarray(tab[col], dtype=float)
    return np.ma.filled(arr, np.nan)


def finite_positive(x):
    x = np.asarray(x, dtype=float)
    return np.isfinite(x) & (x > 0)


def base_quality_mask(tab, y_col=Y_COL, z_col=Z_COL, coverage_min=COVERAGE_MIN):
    y = col_float(tab, y_col)
    z = col_float(tab, z_col)
    mask = np.isfinite(y) & np.isfinite(z)
    if "coverage_signal" in tab.colnames:
        mask &= col_float(tab, "coverage_signal") >= coverage_min
    if "coverage_background" in tab.colnames:
        mask &= col_float(tab, "coverage_background") >= coverage_min
    return mask


def bootstrap_spearman(x, y, n_boot=1000, seed=123):
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    good = np.isfinite(x) & np.isfinite(y)
    x = x[good]
    y = y[good]
    n = len(x)

    if n < 5 or len(np.unique(x)) < 2 or len(np.unique(y)) < 2:
        return np.nan, np.nan, np.nan, np.nan

    rho, p_value = spearmanr(x, y)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        xb = x[idx]
        yb = y[idx]
        if len(np.unique(xb)) < 2 or len(np.unique(yb)) < 2:
            continue
        rb, _ = spearmanr(xb, yb)
        if np.isfinite(rb):
            boots.append(rb)

    if len(boots) == 0:
        return rho, np.nan, np.nan, p_value

    lo, hi = np.nanpercentile(boots, [16, 84])
    return rho, rho - lo, hi - rho, p_value





def binned_spearman_table(
    tab,
    richness_col,
    richness_bins,
    y_col=Y_COL,
    z_col=Z_COL,
    z_bins=Z_BINS,
    n_boot=N_BOOT,
    seed=RNG_SEED,
    coverage_min=COVERAGE_MIN,
):
    lam = col_float(tab, richness_col)
    y = col_float(tab, y_col)
    z = col_float(tab, z_col)
    quality = base_quality_mask(tab, y_col=y_col, z_col=z_col, coverage_min=coverage_min)

    rows = []
    for iz, (zlo, zhi) in enumerate(z_bins):
        for ibin, (llo, lhi) in enumerate(zip(richness_bins[:-1], richness_bins[1:])):
            mask = (
                quality
                & finite_positive(lam)
                & (z >= zlo)
                & (z < zhi)
                & (lam >= llo)
                & (lam < lhi)
            )
            n = int(np.count_nonzero(mask))

            if n >= 5:
                rho, err_lo, err_hi, p_value = bootstrap_spearman(
                    lam[mask], y[mask], n_boot=n_boot, seed=seed + 1000 * iz + ibin
                )
            else:
                rho, err_lo, err_hi, p_value = np.nan, np.nan, np.nan, np.nan

            rows.append(
                {
                    "richness_col": richness_col,
                    "y_col": y_col,
                    "z_low": zlo,
                    "z_high": zhi,
                    "lambda_low": llo,
                    "lambda_high": lhi,
                    "lambda_center": 0.5 * (llo + lhi),
                    "N": n,
                    "spearman_r": rho,
                    "spearman_err_low": err_lo,
                    "spearman_err_high": err_hi,
                    "spearman_p": p_value,
                }
            )

    return Table(rows=rows)


def pretty_quantity_label(col):
    labels = {
        "LAMBDA": r"$\lambda_{\rm RM}$",
        "lambda_spec_true": r"$\lambda_{\rm spec}$",
        "Sigma_excess": r"$\Sigma_{\rm excess}$",
        "N_excess": r"$N_{\rm excess}$",
        "Sigma_signal_covcorr": r"$\Sigma_{\rm signal}$",
        "Sigma_background_covcorr": r"$\Sigma_{\rm bg}$",
    }
    return labels.get(col, col)




def axis_quantity_label(col):
    labels = {
        "Sigma_excess": r"$\Sigma_{\rm excess}$ [galaxies deg$^{-2}$]",
        "Sigma_signal_covcorr": r"$\Sigma_{\rm signal}$ [galaxies deg$^{-2}$]",
        "Sigma_background_covcorr": r"$\Sigma_{\rm bg}$ [galaxies deg$^{-2}$]",
        "N_excess": r"$N_{\rm excess}$",
    }
    return labels.get(col, pretty_quantity_label(col))


def richness_xlim_low(richness_col):
    if richness_col == "LAMBDA":
        return 20.0
    if richness_col == "lambda_spec_true":
        return 4.0
    return None


def richness_xlim_high(richness_col):
    if richness_col == "LAMBDA":
        return 100.0
    if richness_col == "lambda_spec_true":
        return 200.0
    return None

def plot_binned_spearman(summary, richness_col, outfile=None):
    fig, ax = plt.subplots(figsize=(8.0, 5.6))
    colors = sns.color_palette("colorblind", n_colors=len(Z_BINS))

    for color, (zlo, zhi) in zip(colors, Z_BINS):
        mask = (col_float(summary, "z_low") == zlo) & (col_float(summary, "z_high") == zhi)
        sub = summary[mask]
        x = col_float(sub, "lambda_center")
        y = col_float(sub, "spearman_r")
        yerr = np.vstack([
            col_float(sub, "spearman_err_low"),
            col_float(sub, "spearman_err_high"),
        ])

        ax.errorbar(
            x,
            y,
            yerr=yerr,
            marker="o",
            linestyle="-",
            lw=1.6,
            capsize=3,
            color=color,
            label=rf"${zlo:.1f}<z<{zhi:.1f}$",
        )

        for x0, y0, n0 in zip(x, y, np.asarray(sub["N"], dtype=int)):
            if np.isfinite(x0) and np.isfinite(y0):
                ax.text(x0, y0 + 0.035, f"{n0}", fontsize=8, ha="center", va="bottom", color=color)

    ax.axhline(0.0, color="black", lw=1.0, alpha=0.65)
    ax.set_xscale("log")
    ax.set_ylim(-0.2, 0.2)
    ax.set_xlabel(pretty_quantity_label(richness_col))
    ax.set_ylabel(rf"Spearman $\rho$({pretty_quantity_label(richness_col)}, {pretty_quantity_label(Y_COL)})")
    xlo = richness_xlim_low(richness_col)
    xhi = richness_xlim_high(richness_col)
    if xlo is not None or xhi is not None:
        cur_lo, cur_hi = ax.get_xlim()
        ax.set_xlim(xlo if xlo is not None else cur_lo, xhi if xhi is not None else cur_hi)
    ax.legend(frameon=False, loc="upper right")
    style_astronomy_axes(ax)
    fig.tight_layout()

    if outfile is not None:
        fig.savefig(outfile, bbox_inches="tight")
        print("Saved", outfile)

    plt.show()
    return fig, ax

def redshift_bin_spearman_table(
    tab,
    richness_cols=("LAMBDA", "lambda_spec_true"),
    y_col=Y_COL,
    z_col=Z_COL,
    z_bins=Z_BINS,
    n_boot=N_BOOT,
    seed=RNG_SEED,
    coverage_min=COVERAGE_MIN,
):
    """
    Compute one Spearman coefficient per redshift bin, using the full available
    richness range inside that redshift bin. This avoids range restriction from
    binning in richness.
    """
    y = col_float(tab, y_col)
    z = col_float(tab, z_col)
    quality = base_quality_mask(tab, y_col=y_col, z_col=z_col, coverage_min=coverage_min)

    rows = []
    for iz, (zlo, zhi) in enumerate(z_bins):
        zmask = quality & np.isfinite(z) & (z >= zlo) & (z < zhi)
        for j, richness_col in enumerate(richness_cols):
            lam = col_float(tab, richness_col)
            mask = zmask & finite_positive(lam) & np.isfinite(y)
            n = int(np.count_nonzero(mask))

            if n >= 5:
                rho, err_lo, err_hi, p_value = bootstrap_spearman(
                    lam[mask], y[mask], n_boot=n_boot, seed=seed + 1000 * iz + 100 * j
                )
            else:
                rho, err_lo, err_hi, p_value = np.nan, np.nan, np.nan, np.nan

            rows.append(
                {
                    "richness_col": richness_col,
                    "y_col": y_col,
                    "z_low": zlo,
                    "z_high": zhi,
                    "z_center": 0.5 * (zlo + zhi),
                    "N": n,
                    "richness_min": np.nanmin(lam[mask]) if n > 0 else np.nan,
                    "richness_max": np.nanmax(lam[mask]) if n > 0 else np.nan,
                    "spearman_r": rho,
                    "spearman_err_low": err_lo,
                    "spearman_err_high": err_hi,
                    "spearman_p": p_value,
                }
            )
    return Table(rows=rows)


def plot_redshift_bin_spearman(summary, outfile=None):
    fig, ax = plt.subplots(figsize=(7.2, 5.2))
    colors = sns.color_palette("colorblind", n_colors=2)
    richness_cols = ["LAMBDA", "lambda_spec_true"]

    for color, richness_col in zip(colors, richness_cols):
        mask = np.asarray(summary["richness_col"]) == richness_col
        sub = summary[mask]
        x = col_float(sub, "z_center")
        y = col_float(sub, "spearman_r")
        xerr = np.vstack([
            x - col_float(sub, "z_low"),
            col_float(sub, "z_high") - x,
        ])
        yerr = np.vstack([
            col_float(sub, "spearman_err_low"),
            col_float(sub, "spearman_err_high"),
        ])

        ax.errorbar(
            x,
            y,
            xerr=xerr,
            yerr=yerr,
            marker="o",
            linestyle="-",
            lw=1.7,
            capsize=3,
            color=color,
            label=pretty_quantity_label(richness_col),
        )

        for x0, y0, n0 in zip(x, y, np.asarray(sub["N"], dtype=int)):
            if np.isfinite(x0) and np.isfinite(y0):
                ax.text(x0, y0 + 0.015, f"{n0}", fontsize=8, ha="center", va="bottom", color=color)

    ax.axhline(0.0, color="black", lw=1.0, alpha=0.65)
    ax.set_ylim(-0.2, 0.2)
    ax.set_xlabel(r"$z_{\rm spec,BCG}$")
    ax.set_ylabel(rf"Spearman $\rho$(richness, {pretty_quantity_label(Y_COL)})")
    ax.legend(frameon=False, loc="upper right")
    style_astronomy_axes(ax)
    fig.tight_layout()

    if outfile is not None:
        fig.savefig(outfile, bbox_inches="tight")
        print("Saved", outfile)

    plt.show()
    return fig, ax


def adaptive_min_count_bins(values, outer_bins=None, min_per_bin=100, max_bins=7):
    """
    Build adaptive richness bins with at least min_per_bin objects when possible,
    while capping the number of bins at max_bins per redshift slice.

    The number of bins is

        min(max_bins, floor(N / min_per_bin)),

    with a minimum of one bin. Edges are chosen from quantiles of the values so
    the bins have roughly equal occupancy. If the redshift slice has fewer than
    min_per_bin objects, a single bin is returned.
    """
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if outer_bins is not None:
        outer_bins = np.asarray(outer_bins, dtype=float)
        values = values[(values >= outer_bins[0]) & (values < outer_bins[-1])]
        global_lo = outer_bins[0]
        global_hi = outer_bins[-1]
    else:
        if len(values) == 0:
            return np.array([], dtype=float)
        global_lo = np.nanmin(values)
        global_hi = np.nanmax(values)

    values = np.sort(values)
    n = len(values)
    if n == 0:
        return np.array([], dtype=float)

    n_bins = max(1, min(int(max_bins), n // int(min_per_bin)))
    if n_bins == 1:
        return np.array([global_lo, global_hi], dtype=float)

    quantiles = np.linspace(0.0, 1.0, n_bins + 1)
    edges = np.nanquantile(values, quantiles)
    edges[0] = global_lo
    edges[-1] = global_hi
    edges = np.unique(edges)

    if len(edges) < 2:
        return np.array([global_lo, global_hi], dtype=float)
    return edges


def binned_mean_sem_table(
    tab,
    richness_col,
    richness_bins=None,
    y_col=Y_COL,
    z_col=Z_COL,
    z_bins=Z_BINS,
    coverage_min=COVERAGE_MIN,
    adaptive_bins=False,
    min_per_bin=100,
    max_bins=7,
    min_plot_n=20,
):
    """Compute mean and SEM of the local-overdensity statistic in richness/redshift bins."""
    lam = col_float(tab, richness_col)
    y = col_float(tab, y_col)
    z = col_float(tab, z_col)
    quality = base_quality_mask(tab, y_col=y_col, z_col=z_col, coverage_min=coverage_min)

    rows = []
    for zlo, zhi in z_bins:
        zmask = quality & finite_positive(lam) & np.isfinite(y) & (z >= zlo) & (z < zhi)
        if adaptive_bins:
            bins = adaptive_min_count_bins(lam[zmask], outer_bins=richness_bins, min_per_bin=min_per_bin, max_bins=max_bins)
        else:
            bins = np.asarray(richness_bins, dtype=float)
        if len(bins) < 2:
            continue

        for llo, lhi in zip(bins[:-1], bins[1:]):
            mask = zmask & (lam >= llo) & (lam < lhi)
            vals = y[mask]
            lam_vals = lam[mask]
            n = int(np.count_nonzero(np.isfinite(vals)))

            if n == 0:
                mean_y = np.nan
                std_y = np.nan
                sem_y = np.nan
                median_y = np.nan
                lambda_mean = np.nan
                lambda_median = np.nan
            elif n == 1:
                mean_y = float(np.nanmean(vals))
                std_y = np.nan
                sem_y = np.nan
                median_y = float(np.nanmedian(vals))
                lambda_mean = float(np.nanmean(lam_vals))
                lambda_median = float(np.nanmedian(lam_vals))
            else:
                mean_y = float(np.nanmean(vals))
                std_y = float(np.nanstd(vals, ddof=1))
                sem_y = std_y / np.sqrt(n) if n >= min_plot_n else np.nan
                median_y = float(np.nanmedian(vals))
                lambda_mean = float(np.nanmean(lam_vals))
                lambda_median = float(np.nanmedian(lam_vals))

            rows.append(
                {
                    "richness_col": richness_col,
                    "y_col": y_col,
                    "z_low": zlo,
                    "z_high": zhi,
                    "lambda_low": llo,
                    "lambda_high": lhi,
                    "lambda_center": 0.5 * (llo + lhi),
                    "lambda_mean": lambda_mean,
                    "lambda_median": lambda_median,
                    "N": n,
                    "mean_y": mean_y,
                    "std_y": std_y,
                    "sem_y": sem_y,
                    "median_y": median_y,
                    "adaptive_bins": adaptive_bins,
                    "min_per_bin": min_per_bin,
                    "max_bins": max_bins,
                    "min_plot_n": min_plot_n,
                }
            )
    return Table(rows=rows)


def format_spearman_legend(zlo, zhi, spearman_stats):
    rho, rho_lo, rho_hi, _ = spearman_stats

    if np.isfinite(rho) and np.isfinite(rho_lo) and np.isfinite(rho_hi):
        rho_text = rf"$\rho_S={rho:.2f}^{{+{rho_hi:.2f}}}_{{-{rho_lo:.2f}}}$"
    elif np.isfinite(rho):
        rho_text = rf"$\rho_S={rho:.2f}$"
    else:
        rho_text = r"$\rho_S=\mathrm{nan}$"

    return rf"${zlo:.1f}<z<{zhi:.1f}$, {rho_text}"


def style_astronomy_axes(ax):
    """Paper-style axes: boxed frame, inward ticks, no grid."""
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(1.1)
    ax.tick_params(
        axis="both",
        which="both",
        direction="in",
        top=True,
        right=True,
        length=6,
        width=1.0,
    )
    ax.tick_params(which="minor", length=3)
    ax.minorticks_on()


def plot_scatter_with_binned_mean_sem(
    tab,
    richness_col,
    richness_bins=None,
    y_col=Y_COL,
    z_col=Z_COL,
    z_bins=Z_BINS,
    coverage_min=COVERAGE_MIN,
    outfile=None,
    robust_ylim=True,
    adaptive_bins=False,
    min_per_bin=100,
    max_bins=7,
    min_plot_n=20,
    n_boot=N_BOOT,
):
    """Scatter richness vs local overdensity, overlaid with binned mean and SEM."""
    lam = col_float(tab, richness_col)
    y = col_float(tab, y_col)
    z = col_float(tab, z_col)
    quality = base_quality_mask(tab, y_col=y_col, z_col=z_col, coverage_min=coverage_min)

    summary = binned_mean_sem_table(
        tab,
        richness_col=richness_col,
        richness_bins=richness_bins,
        y_col=y_col,
        z_col=z_col,
        z_bins=z_bins,
        coverage_min=coverage_min,
        adaptive_bins=adaptive_bins,
        min_per_bin=min_per_bin,
        max_bins=max_bins,
        min_plot_n=min_plot_n,
    )

    fig, ax = plt.subplots(figsize=(8.0, 5.8))
    colors = sns.color_palette("colorblind", n_colors=len(z_bins))

    for iz, (color, (zlo, zhi)) in enumerate(zip(colors, z_bins)):
        mask = quality & finite_positive(lam) & np.isfinite(y) & (z >= zlo) & (z < zhi)
        spearman_stats = bootstrap_spearman(
            lam[mask], y[mask], n_boot=n_boot, seed=RNG_SEED + 2000 * iz
        )
        label = format_spearman_legend(zlo, zhi, spearman_stats)

        ax.scatter(
            lam[mask],
            y[mask],
            s=12,
            alpha=0.14,
            color=color,
            edgecolor="none",
            rasterized=True,
        )

        smask = (col_float(summary, "z_low") == zlo) & (col_float(summary, "z_high") == zhi)
        sub = summary[smask]
        x_mean = col_float(sub, "lambda_mean")
        y_mean = col_float(sub, "mean_y")
        y_sem = col_float(sub, "sem_y")
        n_bin = np.asarray(sub["N"], dtype=int)
        good = np.isfinite(x_mean) & np.isfinite(y_mean) & np.isfinite(y_sem) & (n_bin >= min_plot_n)

        ax.errorbar(
            x_mean[good],
            y_mean[good],
            yerr=y_sem[good],
            marker="o",
            linestyle="none",
            capsize=3,
            ms=6,
            color=color,
            markeredgecolor="black",
            markeredgewidth=0.5,
            label=label,
        )

    ax.axhline(0.0, color="black", lw=1.0, alpha=0.65)
    ax.set_xscale("log")
    ax.set_xlabel(pretty_quantity_label(richness_col))
    ax.set_ylabel(axis_quantity_label(y_col))
    xlo = richness_xlim_low(richness_col)
    xhi = richness_xlim_high(richness_col)
    if xlo is not None or xhi is not None:
        cur_lo, cur_hi = ax.get_xlim()
        ax.set_xlim(xlo if xlo is not None else cur_lo, xhi if xhi is not None else cur_hi)
    ax.legend(frameon=False, loc="upper right", fontsize=10)

    if robust_ylim:
        valid = quality & finite_positive(lam) & np.isfinite(y)
        if np.count_nonzero(valid) > 10:
            lo, hi = np.nanpercentile(y[valid], [1, 99])
            pad = 0.08 * (hi - lo) if hi > lo else 1.0
            ax.set_ylim(lo - pad, hi + pad)

    style_astronomy_axes(ax)
    fig.tight_layout()

    if outfile is not None:
        fig.savefig(outfile, bbox_inches="tight")
        print("Saved", outfile)

    plt.show()
    return summary, fig, ax


In [ ]:
# Quick data sanity checks
needed = ["LAMBDA", "lambda_spec_true", Z_COL, Y_COL]
missing = [col for col in needed if col not in table.colnames]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

base_mask = base_quality_mask(table)
print(f"Rows: {len(table):,}")
print(f"Rows passing finite/coverage mask: {np.count_nonzero(base_mask):,}")
for col in ["LAMBDA", "lambda_spec_true", Z_COL, Y_COL, "coverage_signal", "coverage_background"]:
    if col in table.colnames:
        x = col_float(table, col)
        print(f"{col:>24s}: finite={np.count_nonzero(np.isfinite(x)):,}, min={np.nanmin(x):.4g}, median={np.nanmedian(x):.4g}, max={np.nanmax(x):.4g}")

In [ ]:
# Show bin counts before plotting correlations
def bin_count_table(tab, richness_col, richness_bins):
    lam = col_float(tab, richness_col)
    z = col_float(tab, Z_COL)
    quality = base_quality_mask(tab)
    rows = []
    for zlo, zhi in Z_BINS:
        for llo, lhi in zip(richness_bins[:-1], richness_bins[1:]):
            mask = quality & finite_positive(lam) & (z >= zlo) & (z < zhi) & (lam >= llo) & (lam < lhi)
            rows.append({"richness_col": richness_col, "z_bin": f"{zlo:.1f}-{zhi:.1f}", "lambda_bin": f"{llo:g}-{lhi:g}", "N": int(np.count_nonzero(mask))})
    return pd.DataFrame(rows)

display(bin_count_table(table, "LAMBDA", LAMBDA_BINS_RM))
display(bin_count_table(table, "lambda_spec_true", LAMBDA_BINS_SPEC))

## Redshift-Controlled Spearman Without Richness Range Restriction

This section computes one Spearman coefficient per redshift bin using the full available richness range inside that bin. This controls redshift coarsely while avoiding the range-restriction problem introduced by binning in richness.

In [ ]:
# Spearman coefficients in redshift bins only: no richness-bin restriction
zbin_summary = redshift_bin_spearman_table(
    table,
    richness_cols=("LAMBDA", "lambda_spec_true"),
    y_col=Y_COL,
    z_bins=Z_BINS,
    n_boot=N_BOOT,
)
display(zbin_summary.to_pandas())

zbin_summary.write(PLOT_DIR / "redshift_only_spearman_full_richness_range.ecsv", format="ascii.ecsv", overwrite=True)
plot_redshift_bin_spearman(
    zbin_summary,
    outfile=PLOT_DIR / "redshift_only_spearman_full_richness_range.png",
);


## Scatter Plots With Binned Mean And SEM

These plots show the individual cluster measurements underneath the binned mean local-overdensity statistic. Error bars show the standard error of the mean,

$$
\mathrm{SEM} = \frac{s}{\sqrt{N}},
$$

computed separately in richness and redshift bins.

In [ ]:
# Scatter plus binned mean and SEM for redMaPPer richness
rm_mean_sem, fig, ax = plot_scatter_with_binned_mean_sem(
    table,
    richness_col="LAMBDA",
    richness_bins=LAMBDA_BINS_RM,
    y_col=Y_COL,
    z_bins=Z_BINS,
    adaptive_bins=False,
    min_per_bin=100,
    min_plot_n=20,
    max_bins=7,
    outfile=PLOT_DIR / "scatter_mean_sem_LAMBDA_vs_local_overdensity.png",
)
display(rm_mean_sem.to_pandas())
rm_mean_sem.write(PLOT_DIR / "scatter_mean_sem_LAMBDA.ecsv", format="ascii.ecsv", overwrite=True)


In [ ]:
# Scatter plus binned mean and SEM for spectroscopic richness
spec_mean_sem, fig, ax = plot_scatter_with_binned_mean_sem(
    table,
    richness_col="lambda_spec_true",
    richness_bins=LAMBDA_BINS_SPEC,
    y_col=Y_COL,
    z_bins=Z_BINS,
    adaptive_bins=False,
    min_per_bin=100,
    min_plot_n=20,
    max_bins=7,
    outfile=PLOT_DIR / "scatter_mean_sem_lambda_spec_true_vs_local_overdensity.png",
)
display(spec_mean_sem.to_pandas())
spec_mean_sem.write(PLOT_DIR / "scatter_mean_sem_lambda_spec_true.ecsv", format="ascii.ecsv", overwrite=True)


In [ ]:
# redMaPPer richness-binned Spearman coefficients
rm_summary = binned_spearman_table(
    table,
    richness_col="LAMBDA",
    richness_bins=LAMBDA_BINS_RM,
    y_col=Y_COL,
    n_boot=N_BOOT,
)
display(rm_summary.to_pandas())

rm_summary.write(PLOT_DIR / "binned_spearman_LAMBDA.ecsv", format="ascii.ecsv", overwrite=True)
plot_binned_spearman(
    rm_summary,
    richness_col="LAMBDA",
    outfile=PLOT_DIR / "spearman_vs_LAMBDA_by_redshift.png",
);

In [ ]:
# Spectroscopic richness-binned Spearman coefficients
spec_summary = binned_spearman_table(
    table,
    richness_col="lambda_spec_true",
    richness_bins=LAMBDA_BINS_SPEC,
    y_col=Y_COL,
    n_boot=N_BOOT,
)
display(spec_summary.to_pandas())

spec_summary.write(PLOT_DIR / "binned_spearman_lambda_spec_true.ecsv", format="ascii.ecsv", overwrite=True)
plot_binned_spearman(
    spec_summary,
    richness_col="lambda_spec_true",
    outfile=PLOT_DIR / "spearman_vs_lambda_spec_true_by_redshift.png",
);